# 🖥️ IT Help Desk Chatbot — Local Testing Notebook

A fully agentic IT helpdesk chatbot featuring:
- **RAG Knowledge Base** (ChromaDB + OpenAI Embeddings) — answers common IT questions with step-by-step guides
- **Ticket Management** (SQLite) — lookup open tickets, detect duplicates, create new tickets
- **MCP Server** (`src/mcp_server.py`) — same tools exposed via the Model Context Protocol
- **LangChain Agent** with full conversation memory

## Architecture
```
User Input
    │
    ▼
ITHelpdeskChatbot (agent.py)
    │
    ├──► search_knowledge_base  ──► ChromaDB (RAG)
    ├──► get_user_info          ──► SQLite users table
    ├──► get_user_tickets       ──► SQLite tickets table
    ├──► search_similar_tickets ──► SQLite tickets table
    └──► create_new_ticket      ──► SQLite tickets table
```

## Sample Global IDs for testing
| Global ID | Name          | Department  |
|-----------|---------------|-------------|
| GID001    | Alice Johnson | Engineering |
| GID002    | Bob Smith     | Engineering |
| GID003    | Carol Davis   | HR          |
| GID004    | David Lee     | Finance     |
| GID005    | Eve Wilson    | IT          |

## 1. Install Dependencies

In [ ]:
%pip install -q langchain langchain-openai langchain-google-genai langchain-community \
    langchain-chroma chromadb openai google-generativeai python-dotenv mcp ipywidgets

## 2. Configuration

Set `LLM_PROVIDER` to `openai` or `gemini`. Supply the matching API key in `.env` or directly below.

| Provider | Required env var | Default model | Default embeddings |
|----------|-----------------|---------------|-------------------|
| `openai` | `OPENAI_API_KEY` | `gpt-4o-mini` | `text-embedding-3-small` |
| `gemini` | `GOOGLE_API_KEY` | `gemini-2.5-flash` | `models/text-embedding-004` |

In [ ]:
import os
import sys
from pathlib import Path



# ── Ensure repo root is importable ────────────────────────────────────────────
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
    print("✓ Loaded .env file")
except ImportError:
    print("⚠️  Not loaded .env file")
    pass

# ── Set API keys here (or in .env) ────────────────────────────────────────────
# os.environ["OPENAI_API_KEY"]  = "sk-..."
# os.environ["GOOGLE_API_KEY"]  = "AIza..."

# ── Provider selection: change to 'gemini' to use Google Gemini ─────────────
# os.environ["LLM_PROVIDER"] = "gemini"

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()

if LLM_PROVIDER == "gemini":
    key   = os.getenv("GOOGLE_API_KEY", "")
    model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
    embed = os.getenv("GEMINI_EMBEDDING_MODEL", "models/text-embedding-004")
    ok    = bool(key) and key != "your_google_api_key_here"
else:
    key   = os.getenv("OPENAI_API_KEY", "")
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    embed = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
    ok    = bool(key) and key != "your_openai_api_key_here"

if ok:
    print(f"✓ Provider: {LLM_PROVIDER.upper()} | model={model} | embeddings={embed}")
else:
    print(f"⚠️  API key for '{LLM_PROVIDER}' is not set. Add it to .env or uncomment the line above.")

✓ Loaded .env file
✓ Provider: GEMINI | model=gemini-2.5-flash | embeddings=models/gemini-embedding-001


## 3. Initialize Database

In [2]:
from src.database import init_db, get_user, get_user_tickets, get_ticket

init_db()
print("✓ Database initialized")

# Quick sanity check
user = get_user("GID001")
print(f"  Sample user: {user['name']} | {user['email']} | {user['department']}")

tickets = get_user_tickets("GID001")
print(f"  Tickets for GID001: {len(tickets)} total")
for t in tickets:
    print(f"    [{t['ticket_id']}] {t['title']} — {t['status']}")

✓ Database initialized
  Sample user: Alice Johnson | alice.johnson@company.com | Engineering
  Tickets for GID001: 2 total
    [TKT-001] Cannot access VPN — In Progress
    [TKT-002] Outlook not syncing emails — Resolved


## 4. Initialize Knowledge Base (ChromaDB + Provider Embeddings)

> First run indexes 12 IT articles. Each provider gets its own ChromaDB collection to avoid embedding dimension mismatches.

In [3]:
import importlib
# !{sys.executable} -m pip install langchain-google-genai --user
# !{sys.executable} -m pip install langchain-openai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from src.knowledge_base import init_knowledge_base, search_kb

try:
    from src.agent import create_embeddings
except (ModuleNotFoundError, ImportError):
    def create_embeddings(provider: str):
        provider = provider.lower()
        if provider == "gemini":
            return GoogleGenerativeAIEmbeddings(model=embed)
        if provider == "openai":
            OpenAIEmbeddings = importlib.import_module("langchain_openai").OpenAIEmbeddings
            return OpenAIEmbeddings(model=embed)
        raise ValueError(f"Unsupported provider: {provider}")

# Each provider uses its own collection to avoid dimension mismatches
collection_name = f"it_knowledge_base_{LLM_PROVIDER}"
embeddings  = create_embeddings(LLM_PROVIDER)
vectorstore = init_knowledge_base(embeddings, collection_name=collection_name)
print(f"  Using collection: {collection_name}")

# Quick test
results = search_kb(vectorstore, "my VPN keeps disconnecting", k=1)
print("\nSample KB search for 'my VPN keeps disconnecting':")
print(f"  → Article: {results[0]['title']} | Score: {results[0]['score']}")
print(f"  Preview: {results[0]['content'][:200]}...")

✓ Indexed 13 KB articles into ChromaDB
  Using collection: it_knowledge_base_gemini

Sample KB search for 'my VPN keeps disconnecting':
  → Article: VPN Connection Issues and Setup | Score: 0.6463
  Preview: The company uses GlobalProtect VPN for secure remote access. Follow these steps to troubleshoot VPN issues:

**Initial VPN Setup:**
1. Download GlobalProtect from the IT portal: https://itportal.compa...


## 5. Set Up LangChain Tools

In [5]:
from src.it_tool import set_vectorstore, get_all_tools

# Inject the vector store so `search_knowledge_base` tool works
set_vectorstore(vectorstore)

tools = get_all_tools()
print(f"✓ {len(tools)} tools registered:")
for t in tools:
    print(f"  • {t.name}")

✓ 6 tools registered:
  • get_user_info
  • get_user_tickets
  • get_ticket_details
  • search_similar_tickets
  • create_new_ticket
  • search_knowledge_base


## 6. Test Individual Tools

Verify each tool works before wiring them into the agent.

In [6]:
from src.it_tool import (
    get_user_info, get_user_tickets, get_ticket_details,
    search_similar_tickets, create_new_ticket, search_knowledge_base
)

# ── Tool: get_user_info ────────────────────────────────────────────────────────
print("=" * 60)
print("TEST: get_user_info(GID001)")
print("=" * 60)
print(get_user_info.invoke({"global_id": "GID001"}))

TEST: get_user_info(GID001)
{
  "global_id": "GID001",
  "name": "Alice Johnson",
  "email": "alice.johnson@company.com",
  "department": "Engineering",
  "manager": "Bob Smith"
}


In [7]:
# ── Tool: get_user_tickets ─────────────────────────────────────────────────────
print("=" * 60)
print("TEST: get_user_tickets(GID001, status=Open)")
print("=" * 60)
print(get_user_tickets.invoke({"global_id": "GID001", "status": ""}))

TEST: get_user_tickets(GID001, status=Open)
Tickets for Alice Johnson (GID001):

  [TKT-001] Cannot access VPN
    Status: In Progress | Priority: High | Category: Network
    Created: 2024-01-15 09:00:00
  [TKT-002] Outlook not syncing emails
    Status: Resolved | Priority: Medium | Category: Email
    Created: 2024-01-10 10:00:00
    Resolution: Cleared Outlook cache and ran the Office repair tool.


In [8]:
# ── Tool: search_similar_tickets ──────────────────────────────────────────────
print("=" * 60)
print("TEST: search_similar_tickets — VPN issue for GID001")
print("=" * 60)
print(search_similar_tickets.invoke({
    "global_id": "GID001",
    "issue_description": "VPN not connecting, authentication fails"
}))

TEST: search_similar_tickets — VPN issue for GID001
Found 1 existing open/in-progress ticket(s) for Alice Johnson:

  [TKT-001] Cannot access VPN
    Status: In Progress | Priority: High
    Opened: 2024-01-15 09:00:00
    Description: VPN connection fails after recent system update. Error: 'Authentication failed'.


In [9]:
# ── Tool: search_knowledge_base ───────────────────────────────────────────────
print("=" * 60)
print("TEST: search_knowledge_base — password reset")
print("=" * 60)
print(search_knowledge_base.invoke({"query": "I forgot my password and cannot login"}))

TEST: search_knowledge_base — password reset
Found 2 relevant KB article(s):

────────────────────────────────────────────────────────────
Article 1: How to Reset Your Password  [Access Management]
────────────────────────────────────────────────────────────
If you have forgotten your password or it has expired, follow these steps:

**Self-Service Password Reset (SSPR):**
1. Navigate to the SSPR portal at https://sspr.company.com
2. Click 'I forgot my password'
3. Enter your company email address (firstname.lastname@company.com)
4. Choose your verification method: SMS to registered mobile, backup email, or Authenticator App
5. Complete the identity verification challenge
6. Enter a new password — requirements: minimum 12 characters, at least one uppercase letter, one lowercase letter, one number, and one special character (!@#$%)
7. Confirm the new password and click 'Finish'
8. You will receive a confirmation email within 2 minutes

**After Reset — If SSO still fails:**
- Clear your b

In [10]:
# ── Tool: get_ticket_details ──────────────────────────────────────────────────
print("=" * 60)
print("TEST: get_ticket_details(TKT-001)")
print("=" * 60)
print(get_ticket_details.invoke({"ticket_id": "TKT-001"}))

TEST: get_ticket_details(TKT-001)
{
  "ticket_id": "TKT-001",
  "global_id": "GID001",
  "title": "Cannot access VPN",
  "description": "VPN connection fails after recent system update. Error: 'Authentication failed'.",
  "category": "Network",
  "priority": "High",
  "status": "In Progress",
  "created_at": "2024-01-15 09:00:00",
  "updated_at": "2024-01-15 14:00:00",
  "resolution": null
}


## 7. Build the IT Helpdesk Agent

In [11]:
from src.agent import create_llm, build_graph, ITHelpdeskChatbot

llm = create_llm(LLM_PROVIDER)
print(f"✓ LLM: {llm.__class__.__name__}")

agent_executor = build_graph(llm, tools)
chatbot        = ITHelpdeskChatbot(agent_executor)

print("✓ IT Helpdesk Agent ready")

✓ LLM: ChatGoogleGenerativeAI
✓ IT Helpdesk Agent ready


## 8. Scripted Scenarios

Run pre-built conversation scenarios to verify the full flow.

In [12]:
def run_scenario(title: str, turns: list[str]) -> None:
    """Run a scripted scenario and pretty-print the conversation."""
    print("\n" + "█" * 70)
    print(f"  SCENARIO: {title}")
    print("█" * 70)
    chatbot.reset()
    for user_msg in turns:
        print(f"\n👤 User : {user_msg}")
        reply = chatbot.chat(user_msg)
        print(f"🤖 Agent: {reply}")
        print("-" * 70)

In [13]:
# ── Scenario 1: Check ticket status ──────────────────────────────────────────
run_scenario(
    "Ticket Status Inquiry",
    [
        "Hi, I'd like to check my open tickets.",
        "My Global ID is GID001",
        "Can you give me details on TKT-001?",
    ]
)


██████████████████████████████████████████████████████████████████████
  SCENARIO: Ticket Status Inquiry
██████████████████████████████████████████████████████████████████████

👤 User : Hi, I'd like to check my open tickets.
🤖 Agent: Hello! To help you with that, please provide your Company Global ID (e.g., GID123).
----------------------------------------------------------------------

👤 User : My Global ID is GID001
🤖 Agent: Thank you, Alice. I am now checking for your open tickets.
----------------------------------------------------------------------

👤 User : Can you give me details on TKT-001?
🤖 Agent: Ticket TKT-001, "Cannot access VPN," is currently In Progress with High priority. It was created on 2024-01-15 and last updated on the same day. The IT team is actively working on this network issue and will follow up with you.
----------------------------------------------------------------------


In [ ]:
# ── Scenario 2: KB lookup → issue resolved, no ticket needed ─────────────────
run_scenario(
    "KB Self-Service — Password Reset",
    [
        "Hello, I forgot my password and I'm locked out.",
        "GID003",
        "Yes, the SSPR portal steps worked. Thanks!",
    ]
)

In [ ]:
# ── Scenario 3: KB lookup → not resolved → duplicate ticket detected ─────────
run_scenario(
    "Duplicate Ticket Detection",
    [
        "I need help, my VPN is not working after the latest update.",
        "GID001",
        "I tried all those steps but VPN still fails. I'd like to raise a ticket.",
    ]
)

In [ ]:
# ── Scenario 4: New issue → no duplicate → create ticket ─────────────────────
run_scenario(
    "New Ticket Creation",
    [
        "My laptop screen is flickering and sometimes goes black.",
        "GID002",
        "Those steps didn't help. Please raise a ticket for me.",
        "Yes, please go ahead and create it.",
    ]
)

In [ ]:
# ── Scenario 5: General IT question (pure KB) ─────────────────────────────────
run_scenario(
    "General IT Question — Teams Audio",
    [
        "How do I fix my microphone not working in Microsoft Teams meetings?",
        "GID005",
        "The device settings fix worked! No ticket needed, thank you.",
    ]
)

## 9. Interactive Chat — ipywidgets UI

A real-time chat interface. Run this cell to open the chat widget.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# ── Reset chatbot for a fresh session ────────────────────────────────────────
chatbot.reset()

# ── Widgets ───────────────────────────────────────────────────────────────────
chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ccc",
        height="450px",
        overflow_y="auto",
        padding="12px",
        background_color="#f9f9f9",
    )
)

input_box = widgets.Text(
    placeholder="Type your message and press Enter…",
    layout=widgets.Layout(width="75%"),
)
send_btn  = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width="12%"),
)
reset_btn = widgets.Button(
    description="New Session",
    button_style="warning",
    layout=widgets.Layout(width="13%"),
)
status_lbl = widgets.Label(value="")


def append_message(role: str, text: str) -> None:
    with chat_output:
        if role == "user":
            print(f"\n👤 You:\n   {text}\n")
        else:
            print(f"🤖 IT Assistant:\n{text}\n")
            print("─" * 68)


def on_send(event=None) -> None:
    msg = input_box.value.strip()
    if not msg:
        return
    input_box.value = ""
    send_btn.disabled = True
    status_lbl.value = "Thinking…"
    append_message("user", msg)
    try:
        reply = chatbot.chat(msg)
    except Exception as exc:
        reply = f"❌ Error: {exc}"
    append_message("assistant", reply)
    send_btn.disabled = False
    status_lbl.value = ""


def on_reset(event=None) -> None:
    chatbot.reset()
    with chat_output:
        from IPython.display import clear_output
        clear_output()
    status_lbl.value = "Session reset."
    welcome = chatbot.chat("Hello")
    append_message("assistant", welcome)
    status_lbl.value = ""


send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
input_box.on_submit(lambda _: on_send())

header  = widgets.HTML("<h3 style='margin:0 0 8px 0'>🖥️ IT Help Desk Chat</h3>")
toolbar = widgets.HBox([input_box, send_btn, reset_btn])
ui      = widgets.VBox([header, chat_output, toolbar, status_lbl])

display(ui)

# ── Auto-greet ────────────────────────────────────────────────────────────────
welcome = chatbot.chat("Hello")
append_message("assistant", welcome)

## 10. MCP Server Configuration

The `src/mcp_server.py` exposes the same tools via the **Model Context Protocol** over stdio.

### Running the MCP server
```bash
cd "C:/Study/AI/IT help/Repo2"
python -m src.mcp_server
```

### Claude Desktop integration
Add to `%APPDATA%\Claude\claude_desktop_config.json`:
```json
{
  "mcpServers": {
    "it-helpdesk": {
      "command": "python",
      "args": ["-m", "src.mcp_server"],
      "cwd": "C:/Study/AI/IT help/Repo2"
    }
  }
}
```

### MCP Tools exposed
| Tool | Description |
|------|-------------|
| `get_user_info` | Look up employee by Global ID |
| `get_user_tickets` | List all tickets for a user |
| `get_ticket_details` | Full details of a ticket |
| `search_similar_tickets` | Find duplicate open tickets |
| `create_ticket` | Create a new IT support ticket |

In [ ]:
# ── Verify MCP server can start (import check only, does not run stdio) ───────

from mcp.server import Server
app = Server("it-helpdesk")
print([x for x in dir(app) if "tool" in x.lower()])
try:
    import importlib, src.mcp_server as mcp_mod
    importlib.reload(mcp_mod)
    print("✓ MCP server module loaded successfully")
    print("  Run 'python -m src.mcp_server' from the repo root to start the MCP server.")
except Exception as e:
    print(f"❌ MCP server import error: {e}")

## 11. Streamlit UI — Migration Path

Create `streamlit_app.py` in the repo root to get a production-grade web UI:

In [ ]:
STREAMLIT_TEMPLATE = '''
"""Streamlit IT Help Desk Chatbot — run with: streamlit run streamlit_app.py"""
import os, sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

import streamlit as st
from dotenv import load_dotenv

from src.database import init_db
from src.knowledge_base import init_knowledge_base
from src.it_tools import set_vectorstore, get_all_tools
from src.agent import create_llm, create_embeddings, build_agent, ITHelpdeskChatbot

load_dotenv()

st.set_page_config(page_title="IT Help Desk", page_icon="🖥️", layout="centered")
st.title("🖥️ IT Help Desk Assistant")

@st.cache_resource
def init_chatbot():
    provider = os.getenv("LLM_PROVIDER", "gemini").lower()
    init_db()
    embeddings  = create_embeddings(provider)
    vectorstore = init_knowledge_base(embeddings, collection_name=f"it_knowledge_base_{provider}")
    set_vectorstore(vectorstore)
    tools    = get_all_tools()
    llm      = create_llm(provider)
    executor = build_agent(llm, tools)
    return ITHelpdeskChatbot(executor)

chatbot = init_chatbot()

if "messages" not in st.session_state:
    st.session_state.messages = []
    welcome = chatbot.chat("Hello")
    st.session_state.messages.append({"role": "assistant", "content": welcome})

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Type your message…"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Thinking…"):
            reply = chatbot.chat(prompt)
        st.markdown(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})

if st.sidebar.button("🔄 New Session"):
    chatbot.reset()
    st.session_state.messages = []
    st.rerun()
'''

streamlit_path = Path("streamlit_app.py")
if not streamlit_path.exists():
    streamlit_path.write_text(STREAMLIT_TEMPLATE.strip())
    print("✓ streamlit_app.py created")
    print("  Install Streamlit: pip install streamlit")
    print("  Run: streamlit run streamlit_app.py")
else:
    print("  streamlit_app.py already exists")

---
## Quick Reference

| Action | Command |
|--------|---------|
| Run notebook | `jupyter notebook it_helpdesk_chatbot.ipynb` |
| Start MCP server | `python -m src.mcp_server` |
| Launch Streamlit UI | `streamlit run streamlit_app.py` |
| Reset DB | Delete `data/tickets.db` and re-run init cell |
| Rebuild KB index | Delete `data/chroma_db/` and re-run init cell |

### Agent Conversation Flow
```
1. Ask for Global ID → validate user
2. Describe issue   → KB search (RAG) → present steps
3. If unresolved    → check duplicate tickets
   ├─ Duplicate found  → show existing ticket, offer to track
   └─ No duplicate     → confirm → create new ticket
```

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("data/tickets.db")
print("── USERS ──")
display(pd.read_sql("SELECT * FROM users", con))
print("── TICKETS ──")
display(pd.read_sql("SELECT * FROM tickets ORDER BY created_at DESC", con))
con.close()

In [ ]:
import sqlite3

con = sqlite3.connect("data/tickets.db")
cur = con.cursor()

print("── USERS ──")
for row in cur.execute("SELECT * FROM users"):
    print(row)

print("\n── TICKETS ──")
for row in cur.execute(
    "SELECT * FROM tickets ORDER BY created_at DESC"
):
    print(row)

con.close()